### Tutorial Walks in BuilDyn

The FMUs in BuilDyn can be changed (for instance the parameter). However, one parameter might influence others. For instance, the U-value of walls in a building may influence the total heating power of the buildings heat source.

For this scenario, we introduce the Variator class (inspired from BuilDa) to BuilDyn. The variator takes FMU variables that will be re-set and changes them accordingly when the initial variables of the FMU are set. 


We again iniitialize a FMU object as before.

In [ ]:
from buildyn import FMU
import json

# Example FMU from BuilDa 2.0
fmu_path = "resources/building.fmu"

# Initial variables from the BuilDa 2.0 FMU -> Needs extra variables here because the BuilDa FMU is weird. For other use-cases you don't have to do this.
with open("resources/fmu_state_dict.json", "r") as f:
    start_variables = json.load(f)

start_variables.update({
    "weaDat.filNam": "resources/Munich.mos",
    "internalGain.fileName": "resources/NoActivity.txt",
    "hygienicalWindowOpening.fileName": "resources/no_opening.txt",
    "UseInternalController.k": 0
})

# FMU object from the buildyn package
fmu = FMU(fmu_file=fmu_path, init_values=start_variables)

Now we initialize the Converter class that handles parameter conversion. We add the converter function from Builda 2.0, but these can vary from FMU to FMU.

In [ ]:
from buildyn.converter import Converter
from buildyn.examples.builda.converter_functions.Component_configurator import Component_configurator
from buildyn.examples.builda.converter_functions.Link_resolver import Link_resolver
from buildyn.examples.builda.converter_functions.Miscellaneous_handler import Miscellaneous_handler
from buildyn.examples.builda.converter_functions.Model_compatibility_layer import Model_compatibility_layer
from buildyn.examples.builda.converter_functions.Nominal_cooling_power_calculator import Nominal_cooling_power_calculator
from buildyn.examples.builda.converter_functions.Nominal_heating_power_calculator import Nominal_heating_power_calculator
from buildyn.examples.builda.converter_functions.RC_Distribution_Configurator import RC_Distribution_Configurator
from buildyn.examples.builda.converter_functions.Zone_dimensions_calculator import Zone_dimensions_calculator
from buildyn.examples.builda.converter_functions.Component_properties_calculator import Component_properties_calculator

converter = Converter()

# All the converter functiosn from BuilDa 2.0
converter_functions = [Link_resolver, Miscellaneous_handler, Model_compatibility_layer, Zone_dimensions_calculator, Component_configurator, RC_Distribution_Configurator, Component_properties_calculator, Nominal_heating_power_calculator, Nominal_cooling_power_calculator]

for cf in converter_functions:
    converter.add_converter_function(cf())

The calculations in the converter may need variables / values that do not exist in the FMU itself. For instance, Changing the wall U-value needs a parameter n_floors that is not explicitly available in the FMU. For this case, we added converter_variables to the converter. These will be used in the conversion, but never set in the FMU itself.

In [ ]:
converter_variables = {
  "zone_length": 9.4125,
  "zone_width": 8,
  "floor_height": 3.04,
  "n_floors": 3,

  "fAWin_south": 0.14056,
  "fAWin_west": 0.14056,
  "fAWin_north": 0.14056,
  "fAWin_east": 0.14056,

  "fATransToAWindow": 0.9,
  "fARoofToAFloor": 1.63612217795485,
  "fAInt": 1.238235294117647,

  "heatCapacity_furniture_per_m2": 2230,

  "UExt": 0.665,
  "heatCapacity_wall": 192000,

  "UFloor": 0.514,
  "heatCapacity_floor": 483840,

  "UInt": 1,
  "heatCapacity_internalWall": 145154,

  "URoof": 0.402,
  "heatCapacity_roof": 81240,

  "UWin": 3.2,
  "thermalZone.gWin": 0.7,

  "heatRecoveryRate": 0,
  "airChangeRate": 0.3,

  "heatingCurve_steepness": 1,
  "relative_heatPump_efficiency": 0.8,

  "internalGainsConvectiveFraction": 0.4,
  "heatingConvectiveFraction": 1,

  "weaDat.fileName": "resources/Munich.mos",
  "internalGain.fileName": "resources/NoActivity.txt",
  "hygienicalWindowOpening.fileName": "resources/no_opening.txt",

  "roomTempLowerSetpoint": 18,
  "roomTempUpperSetpoint": 22,
  "UseInternalController": 0,

  "extWall_C_distribution": "monolythic",
  "floor_C_distribution": [478800, 5040, 0.001],
  "roof_C_distribution": [22440, 58800, 0.001],

  "extWall_R_distribution": "monolythic",
  "floor_R_distribution": [0.0525, 0.0525, 0.1579, 0.0001],
  "roof_R_distribution": [1.1085, 1.1085, 0.1, 0.0001],

  "intWall_R_distribution": "monolythic",
  "intWall_C_distribution": "monolythic",

  "Rsi_extWall": 0.13333333333333333,
  "Rsi_intWall": 0.13333333333333333,
  "Rsi_floor": 0.17543859649122806,
  "Rsi_roof": 0.1,
  "Rse_extWall": 0.04,
  "Rse_roof": 0.04,
  "Rsi_window": 0.13333333333333333,
  "Rse_window": 0.05,

  "ta_min": None,
  "ti_set": None
}

# Set the converter variables in the converter. This also updates previously set variables.
converter.update_converter_variables(converter_variables)

Now with the FMU we observe that the heating power changes when the UExt changes.

In [ ]:
fmu_wo_converter = fmu.__copy__()
fmu_w_converter = fmu.__copy__()

# Init the converter in the FMU
fmu_w_converter.set_converter(converter=converter)

# Set the UExt in both FMUs
fmu_wo_converter.set_initial_variables({"UExt": 1.5})
fmu_w_converter.set_initial_variables({"UExt": 0.665})

print(f"Heating Power FMU no converter: {fmu_wo_converter.get_variable('heatingPower')}")
print(f"Heating Power FMU with converter: {fmu_w_converter.get_variable('heatingPower')}")

We can also see the differences in behavior (of the heating source) when we simulate the timeseries with heating source set to always 100%.

In [ ]:
# Set heating source signal always to 1.
fmu_wo_converter.set_variable("ctrSignalHeating", 1)
fmu_w_converter.set_variable("ctrSignalHeating", 1)

df1 = fmu_wo_converter.simulate(observables=["thermalZone.TAir"])
df2 = fmu_w_converter.simulate(observables=["thermalZone.TAir"])

# We see minor differences here.
df1["thermalZone.TAir"].plot()
df2["thermalZone.TAir"].plot()

Because this procedure of creating the BUilDa FMU fully configured each time is tideous, we created a helper function for that.

In [ ]:
from builda_fmu import get_configured_builda_fmu

builda_fmu = get_configured_builda_fmu()

builda_fmu.simulate(observables=["thermalZone.TAir"])["thermalZone.TAir"].plot()